<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Cut Your Vector Database Bill by Dropping JSON**


Estimated time needed: **45** minutes


Most teams building a search system over their own documents write the same line without thinking
about it:

```python
vector = embed(json.dumps(record))
```

It looks tidy. It is also the most common inefficiency in production retrieval systems, and it hides
well, because nothing breaks. Search works, results come back, and the bill is quietly two to three
times larger than it needs to be.

In this lab you will take a product-support knowledge base, measure exactly what those braces and
field names cost, rewrite the text that gets embedded, and prove with a retrieval evaluation that
your cheaper version is just as accurate. Along the way you will find one result that contradicts
the obvious conclusion, and learning to spot that is the real point of the exercise.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
            <li><a href="#Importing-Required-Libraries">Importing Required Libraries</a></li>
            <li><a href="#Defining-Helper-Functions">Defining Helper Functions</a></li>
        </ol>
    </li>
    <li>
        <a href="#Background">Background</a>
        <ol>
            <li><a href="#What-does-an-embedding-model-see?">What does an embedding model see?</a></li>
            <li><a href="#Where-does-the-rest-of-the-record-go?">Where does the rest of the record go?</a></li>
        </ol>
    </li>
    <li><a href="#Example-1---One-record,-two-ways">Example 1: One record, two ways</a></li>
    <li>
        <a href="#Example-2---Measuring-the-whole-knowledge-base">Example 2: Measuring the whole knowledge base</a>
        <ol>
            <li><a href="#The-wrapper-comes-out-of-your-content-budget">The wrapper comes out of your content budget</a></li>
            <li><a href="#What-it-costs-at-scale">What it costs at scale</a></li>
        </ol>
    </li>
    <li>
        <a href="#Exercises">Exercises</a>
        <ol>
            <li><a href="#Exercise-1---Write-your-own-formatter">Exercise 1: Write your own formatter</a></li>
            <li><a href="#Exercise-2---Split-the-record">Exercise 2: Split the record</a></li>
            <li><a href="#Exercise-3---Prove-it-does-not-hurt-retrieval">Exercise 3: Prove it does not hurt retrieval</a></li>
        </ol>
    </li>
</ol>


## Objectives

After completing this lab you will be able to:

 - Explain what an embedding model does with the braces, quotes, and field names in a JSON string.
 - Measure the token, cost, and truncation difference between JSON and plain text on a real corpus.
 - Decide which fields belong inside the embedded text and which belong in metadata columns.
 - Evaluate retrieval quality with recall@k and MRR, and use it to check that an optimization is free.
 - Recognize the cases where keeping JSON inside the embedded text is still the right call.


----


## Setup


For this lab, we will be using the following libraries:

*   [`numpy`](https://numpy.org/) for vector arithmetic and cosine similarity.
*   [`pandas`](https://pandas.pydata.org/) for displaying measurement tables.
*   [`scikit-learn`](https://scikit-learn.org/stable/) for a small local embedding model that runs
    without an API key.
*   [`matplotlib`](https://matplotlib.org/) for the token breakdown chart.
*   [`tiktoken`](https://github.com/openai/tiktoken) for counting tokens the way an encoder does.


### Installing Required Libraries

The following libraries are not all pre-installed in the Skills Network Labs environment.
__You will need to run the following cell__ to install them.


In [ ]:
%pip install -q numpy~=2.4 pandas~=3.0 scikit-learn~=1.8 matplotlib~=3.10 tiktoken~=0.14

### Importing Required Libraries

_Import all required libraries in one place:_


In [ ]:
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

# Suppress warnings generated by the code below:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

pd.set_option("display.max_colwidth", 60)
plt.rcParams["figure.figsize"] = (8, 4)
print("Success")

### Defining Helper Functions


Three pieces of machinery are needed before we can measure anything: a token counter, an embedding
model, and a retrieval scorer, we will build them here. Then we will use a mock vector database with a lot of mock data. Each helper function has it's description of what it should do.


In [ ]:
# --- 1. Counting tokens -----------------------------------------------------
# An encoder bills and truncates by token, so token count is the unit that matters.
# We use tiktoken when its vocabulary is reachable, and a fitted approximation when it is not.
_PATTERN = re.compile(r"'\w{1,2}| ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")

try:
    import tiktoken
    _ENCODING = tiktoken.get_encoding("cl100k_base")
    TOKENIZER_NAME = "tiktoken cl100k_base (exact)"
except Exception:
    _ENCODING = None
    TOKENIZER_NAME = "byte-pair approximation (within about 3%)"


def token_pieces(text):
    """Split text the way a byte-pair encoder would."""
    if _ENCODING is not None:
        return [_ENCODING.decode([t]) for t in _ENCODING.encode(text)]
    out = []
    for piece in _PATTERN.findall(text):
        stripped = piece.strip()
        if not stripped:
            n = max(1, math.ceil(len(piece) / 2))       # whitespace and indentation
        elif stripped.isalpha():
            n = max(1, math.ceil(len(stripped) / 8))    # a word
        else:
            n = max(1, math.ceil(len(stripped) / 3))    # numbers and punctuation runs
        size = math.ceil(len(piece) / n)
        out.extend(piece[i:i + size] for i in range(0, len(piece), size))
    return out


def count_tokens(text):
    """Counts the number of tokens in a piece of text to estimate token usage."""
    return len(token_pieces(text))


def syntax_share(text):
    """Fraction of tokens with no letters in them: braces, quotes, colons, indentation."""
    pieces = token_pieces(text)
    if not pieces:
        return 0.0
    return sum(1 for p in pieces if not re.search(r"[A-Za-z]{2,}", p)) / len(pieces)
print("Success")

In [ ]:
# --- 2. A small embedding model --------------------------------------------
# A real encoder (watsonx.ai, for instance) would be a drop-in replacement for this class.
# What matters for the lab is that the SAME model scores every format, so the comparison is fair.
class SimpleEmbedder:
    """Converts text into normalized numerical vectors using TF-IDF and SVD, providing a lightweight embedding model that runs locally without an external AI service."""

    def __init__(self, dim=128):
        self.dim = dim
        self._fitted = None

    def encode(self, texts):
        texts = list(texts)
        if self._fitted is None:
            vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words="english",
                                         ngram_range=(1, 2), min_df=1)
            matrix = vectorizer.fit_transform(texts)
            k = min(self.dim, matrix.shape[1] - 1, max(2, matrix.shape[0] - 1))
            svd = TruncatedSVD(n_components=k, random_state=0).fit(matrix)
            self._fitted = (vectorizer, svd)
            vectors = svd.transform(matrix)
        else:
            vectorizer, svd = self._fitted
            vectors = svd.transform(vectorizer.transform(texts))
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        return vectors / np.clip(norms, 1e-9, None)
print("Success")

In [ ]:
# --- 3. Scoring retrieval ---------------------------------------------------
def evaluate_retrieval(texts, doc_ids, questions, k=3):
    """Embed the corpus, run every question, report recall@1, recall@k and MRR."""
    model = SimpleEmbedder()
    doc_vectors = model.encode(texts)
    query_vectors = model.encode([q["question"] for q in questions])

    ranks = []
    for i, q in enumerate(questions):
        order = np.argsort(-(doc_vectors @ query_vectors[i]))
        ranked_ids = [doc_ids[j] for j in order]
        ranks.append(ranked_ids.index(q["answer"]) + 1 if q["answer"] in ranked_ids else 0)

    hit = lambda n: sum(1 for r in ranks if 0 < r <= n) / len(ranks)
    return {"recall@1": hit(1),
            f"recall@{k}": hit(k),
            "MRR": sum(1 / r for r in ranks if r > 0) / len(ranks)}


def plot_tokens(counts, title="Where the tokens go in one chunk"):
    """Stacked bar chart of content tokens versus wrapper tokens."""
    names = list(counts)
    content = min(counts.values())
    wrapper = [counts[n] - content for n in names]
    fig, ax = plt.subplots()
    ax.bar(names, [content] * len(names), label="content", color="#4a6fa5")
    ax.bar(names, wrapper, bottom=content, label="wrapper", color="#d98b5f")
    for i, n in enumerate(names):
        ax.text(i, counts[n] + 3, str(round(counts[n])), ha="center")
    ax.set_ylabel("tokens per chunk")
    ax.set_title(title)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()
print("Success")

In [ ]:
# --- 4. The data ------------------------------------------------------------
# A product-support knowledge base, with the kind of metadata a real content system attaches.
KNOWLEDGE_BASE = [   {   'doc_id': 'KB-1001',
        'title': 'Reset a forgotten account password',
        'product': 'Cloud Pak for Data',
        'version': '5.2',
        'category': 'Account management',
        'author': 'a.okafor',
        'last_updated': '2026-01-14',
        'tags': ['password', 'login', 'account'],
        'body': 'If you cannot sign in, select Forgot password on the login screen and enter the '
                'email address tied to your account. A reset link arrives within five minutes and '
                'stays valid for one hour. The new password must be at least twelve characters and '
                'cannot repeat any of your previous five passwords. If no email arrives, check '
                'your spam folder and confirm with your administrator that your account is still '
                'active.',
        'author_email': 'a.okafor@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1001',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1002',
        'title': 'Enable two-factor authentication',
        'product': 'Cloud Pak for Data',
        'version': '5.2',
        'category': 'Account management',
        'author': 'a.okafor',
        'last_updated': '2026-02-03',
        'tags': ['2fa', 'security', 'login'],
        'body': 'Open Profile and settings, then choose Security. Select Add verification method '
                'and scan the QR code with any authenticator application. Store the ten recovery '
                'codes somewhere safe, because they are the only way back in if you lose the '
                'device. Administrators can require two-factor authentication for every member of '
                'a workspace from the account policy page.',
        'author_email': 'a.okafor@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1002',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1003',
        'title': 'Increase the vector index memory limit',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Performance',
        'author': 'l.moreau',
        'last_updated': '2026-02-20',
        'tags': ['milvus', 'memory', 'index'],
        'body': 'A vector collection loads its index into memory before it can serve queries. When '
                'the collection grows past the node memory limit, load operations fail with a '
                'resource exhausted error. Raise the limit in the query node configuration, or '
                'shard the collection across additional nodes. Reducing the stored payload per '
                'record also helps, because the raw text travels with every search result.',
        'author_email': 'l.moreau@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1003',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1004',
        'title': 'Choose a distance metric for similarity search',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Search quality',
        'author': 'l.moreau',
        'last_updated': '2026-01-28',
        'tags': ['cosine', 'index', 'tuning'],
        'body': 'Cosine similarity compares direction and ignores magnitude, which suits '
                'normalized sentence embeddings. Inner product is faster when vectors are already '
                'unit length, and returns identical rankings in that case. Euclidean distance is '
                'rarely the right choice for text retrieval. Whichever you pick, the metric used '
                'at query time must match the one the index was built with, or results will look '
                'random.',
        'author_email': 'l.moreau@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1004',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1005',
        'title': 'Fix slow ingestion when loading documents',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Performance',
        'author': 'd.singh',
        'last_updated': '2026-03-05',
        'tags': ['ingest', 'batch', 'throughput'],
        'body': 'Ingestion slows down when records are inserted one at a time. Batch between one '
                'hundred and one thousand records per request so the client spends less time '
                'waiting on network round trips. Build the index after the bulk load rather than '
                'during it. Oversized text payloads are another common cause: trimming boilerplate '
                'from each record cuts both the embedding call and the write.',
        'author_email': 'd.singh@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1005',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1006',
        'title': 'Understand embedding token limits',
        'product': 'watsonx.ai',
        'version': '2026.1',
        'category': 'Embeddings',
        'author': 'd.singh',
        'last_updated': '2026-02-11',
        'tags': ['tokens', 'truncation', 'limits'],
        'body': 'Encoder models accept a fixed number of tokens, commonly five hundred and twelve. '
                'Text beyond that limit is silently discarded before the vector is produced, so '
                'the end of a long chunk never influences the result. Count tokens before you send '
                'a batch. If chunks regularly approach the ceiling, shorten the wrapper around the '
                'content rather than cutting the content itself.',
        'author_email': 'd.singh@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1006',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1007',
        'title': 'Pick a chunk size for retrieval',
        'product': 'watsonx.ai',
        'version': '2026.1',
        'category': 'Embeddings',
        'author': 'm.rossi',
        'last_updated': '2026-01-09',
        'tags': ['chunking', 'rag', 'tuning'],
        'body': 'Short chunks give precise matches but lose surrounding context. Long chunks keep '
                'context but blur the signal, because a single vector must represent several ideas '
                'at once. Most question answering systems land between two hundred and five '
                'hundred tokens with a small overlap. Measure recall on real questions before '
                'committing, and re-measure whenever the chunking rules change.',
        'author_email': 'm.rossi@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1007',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1008',
        'title': 'Filter results by metadata fields',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Search quality',
        'author': 'm.rossi',
        'last_updated': '2026-02-25',
        'tags': ['filter', 'metadata', 'query'],
        'body': 'Store attributes such as product, version, language and access level as scalar '
                'fields on the record rather than inside the embedded text. The engine can then '
                'apply an exact filter before or alongside the vector search. Filtering on a field '
                'is precise and cheap, while hoping the embedding captured that attribute is '
                'neither.',
        'author_email': 'm.rossi@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1008',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1009',
        'title': 'Reduce the cost of an embedding pipeline',
        'product': 'watsonx.ai',
        'version': '2026.1',
        'category': 'Cost',
        'author': 'j.tanaka',
        'last_updated': '2026-03-12',
        'tags': ['cost', 'tokens', 'pipeline'],
        'body': 'Embedding providers bill per input token, so the bill follows the text you send, '
                'not the number of records. Remove markup, repeated headers and serialization '
                'syntax before encoding. Cache vectors by a hash of the exact input string so '
                'unchanged content is never paid for twice. Re-embedding an entire corpus after a '
                'formatting change is the most common source of surprise spend.',
        'author_email': 'j.tanaka@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1009',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1010',
        'title': 'Handle duplicate records in a collection',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Data quality',
        'author': 'j.tanaka',
        'last_updated': '2026-01-22',
        'tags': ['duplicates', 'dedupe', 'upsert'],
        'body': 'Near duplicate records crowd out diverse results, since several slots in the top '
                'ten describe the same thing. Deduplicate on a content hash at ingestion and '
                'prefer upsert over insert so a repeated load replaces rather than appends. A '
                'similarity threshold sweep over an existing collection will surface clusters that '
                'slipped through.',
        'author_email': 'j.tanaka@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1010',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1011',
        'title': 'Connect a client application to the service',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Getting started',
        'author': 'a.okafor',
        'last_updated': '2026-02-07',
        'tags': ['sdk', 'connection', 'setup'],
        'body': 'Create an API key from the service instance page and store it in an environment '
                'variable rather than in source control. The client needs the host, the port and '
                'the collection name. Connections are pooled, so build the client once at process '
                'start and reuse it. A handshake failure almost always means the key lacks the '
                'reader role on that instance.',
        'author_email': 'a.okafor@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1011',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1012',
        'title': 'Interpret a low similarity score',
        'product': 'watsonx.ai',
        'version': '2026.1',
        'category': 'Search quality',
        'author': 'l.moreau',
        'last_updated': '2026-03-01',
        'tags': ['scoring', 'relevance', 'debug'],
        'body': 'Cosine scores are only comparable within one model and one collection. A top '
                'score near 0.35 does not mean the answer is wrong; it often means the query and '
                'the passage use different vocabulary. Inspect the retrieved text before tuning '
                'thresholds. Absolute cutoffs tend to age badly, so prefer taking the top few '
                'results and letting a reranker or the language model decide.',
        'author_email': 'l.moreau@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1012',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1013',
        'title': 'Back up and restore a collection',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Operations',
        'author': 'd.singh',
        'last_updated': '2026-02-18',
        'tags': ['backup', 'restore', 'operations'],
        'body': 'A backup captures the vectors, the scalar fields and the collection schema. '
                'Restores must target the same dimension and distance metric, otherwise the index '
                'rebuild fails. Because vectors can be regenerated from the source text, many '
                'teams back up the source and the schema only, and accept a rebuild window in '
                'exchange for a much smaller snapshot.',
        'author_email': 'd.singh@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1013',
        'access_level': 'public',
        'schema_version': '1.4.0'},
    {   'doc_id': 'KB-1014',
        'title': 'Troubleshoot empty search results',
        'product': 'watsonx.data',
        'version': '2.4',
        'category': 'Search quality',
        'author': 'm.rossi',
        'last_updated': '2026-03-08',
        'tags': ['debug', 'empty', 'query'],
        'body': 'An empty result set usually has a boring cause. Confirm the collection is loaded, '
                'that the filter expression matches at least one record, and that the query vector '
                'has the same dimension as the index. Mixing embedding models between ingestion '
                'and query time produces results that are empty or nonsensical rather than merely '
                'poor.',
        'author_email': 'm.rossi@example.com',
        'source_url': 'https://docs.example.com/kb/kb-1014',
        'access_level': 'public',
        'schema_version': '1.4.0'}]

# Real questions, each paired with the article that should answer it.
QUESTIONS = [   {'question': 'I forgot my password, how do I get back in?', 'answer': 'KB-1001'},
    {'question': 'how do I turn on 2fa', 'answer': 'KB-1002'},
    {'question': 'my collection fails to load with a memory error', 'answer': 'KB-1003'},
    {'question': 'should I use cosine or inner product', 'answer': 'KB-1004'},
    {'question': 'loading documents is really slow', 'answer': 'KB-1005'},
    {'question': 'what happens when text is longer than the model allows', 'answer': 'KB-1006'},
    {'question': 'how big should my chunks be', 'answer': 'KB-1007'},
    {'question': 'can I restrict search to one product version', 'answer': 'KB-1008'},
    {'question': 'our embedding bill is too high', 'answer': 'KB-1009'},
    {'question': 'the same document keeps appearing several times', 'answer': 'KB-1010'},
    {'question': 'where do I get an api key to connect', 'answer': 'KB-1011'},
    {'question': 'is a similarity score of 0.3 bad', 'answer': 'KB-1012'},
    {'question': 'how do I snapshot and restore my data', 'answer': 'KB-1013'},
    {'question': 'search returns nothing at all', 'answer': 'KB-1014'}]

print(f"{len(KNOWLEDGE_BASE)} articles, {len(QUESTIONS)} labelled questions")
print("tokenizer:", TOKENIZER_NAME)

print("Success")

## Background

Two seperate ideas to understand:


### What does an embedding model see?

Your code sees `{"title": "Reset your password", "doc_id": "KB-1001"}` as an object with fields. An embedding model has no parser; therefore, it sees a sequence of tokens and produces a single vector that summarizes all of them.

That has two consequences:

1. **All non-alphabetic characters, such as braces, quotation marks, and colons, count as tokens** and are billed at the same rate as words.
2. **Duplicated data doesn't help semantic search.** Fields such as `doc_id`, `schema_version`, and `access_level` appear in all 10,000 chunks. Since these fields are identical in every record, they provide no useful information for distinguishing one chunk from another.


### Where does the rest of the record go?

The fix is not to throw the metadata away. Every serious vector database like Milvus, the vector
features in watsonx.data, Elasticsearch, pgvector — stores scalar fields beside each vector and can
filter on them directly: `product = "watsonx.data"`, `access_level = "public"`,
`last_updated > "2026-01-01"`.

A filter on a field is exact and cheap. Hoping that an embedding captured a date is neither. So one
record becomes two things, and the rule for splitting them is short:

> **Embed the meaning. Store the rest.**


## Example 1 - One record, two ways

Your company runs a support site. A learner types "I forgot my password" into the search box, and
somewhere behind it, an article has been turned into a vector. Let's look at the article the way the
pipeline currently sends it, and the way a human would write it.


In [ ]:
record = KNOWLEDGE_BASE[0]

def as_json(rec):
    """What most pipelines embed: the whole record, serialized."""
    return json.dumps(rec, indent=2)

def as_plain(rec):
    """A title, one line of context, and the body. Nothing else."""
    return f"{rec['title']}\n{rec['product']} {rec['version']} - {rec['category']}\n\n{rec['body']}"

print("--- AS JSON ---")
print(as_json(record)[:600], "...\n")
print("--- AS PLAIN TEXT ---")
print(as_plain(record))


Both strings describe the same article. Now look at what the tokenizer does to each one. The cell
below prints the first thirty tokens exactly as the model receives them.


In [ ]:
def show(text, n=30):
    return " | ".join(p.replace("\n", "\\n") for p in token_pieces(text)[:n])

print("JSON :", show(as_json(record)))
print()
print("PLAIN:", show(as_plain(record)))
print()
print(f"JSON  total: {count_tokens(as_json(record)):>4} tokens")
print(f"PLAIN total: {count_tokens(as_plain(record)):>4} tokens")

Read the JSON line carefully. Roughly the first two dozen tokens go by before a single word of the
actual answer appears: `{`, a newline, two spaces, a quote, `doc`, `_`, `id`, `":`. You are paying
for punctuation, and a user searching for "I forgot my password" will never type any of it.


## Example 2 - Measuring the whole knowledge base

One record could be a fluke. Let's measure all fourteen, in four different representations: indented
JSON, compact JSON, plain text, and body text with no header at all.


In [ ]:
def as_compact_json(rec):
    return json.dumps(rec, separators=(",", ":"))

def as_body_only(rec):
    return rec["body"]

FORMATS = {
    "json_indented": as_json,
    "json_compact": as_compact_json,
    "plain_text": as_plain,
    "body_only": as_body_only,
}

rows = []
rendered = {}
for name, fn in FORMATS.items():
    texts = [fn(rec) for rec in KNOWLEDGE_BASE]
    rendered[name] = texts
    rows.append({
        "format": name,
        "tokens": round(np.mean([count_tokens(t) for t in texts])),
        "bytes": round(np.mean([len(t.encode()) for t in texts])),
        "syntax": f"{np.mean([syntax_share(t) for t in texts]) * 100:.0f}%",
    })

measurements = pd.DataFrame(rows).set_index("format")
baseline = measurements.loc["json_indented", "tokens"]
measurements["saving"] = [f"{(1 - t / baseline) * 100:.0f}%" for t in measurements["tokens"]]
measurements

In [ ]:
plot_tokens({row.Index: row.tokens for row in measurements.itertuples()})

The `syntax` column counts only tokens with no letters in them — braces, quotes, colons, indentation.
In the indented-JSON row that is already more than half of everything you pay to embed, and it is an
undercount: repeated field names like `"title"` and `"schema_version"` are structure too, and they are
not included in that figure.


### The wrapper comes out of your content budget

Encoder models accept a fixed number of tokens — 512 for IBM's Granite and slate embedding models,
256 for some smaller ones. Text past the limit is not an error. It is **silently discarded** before
the vector is computed, and nothing in your logs will mention it. You would need to split the chunks.

The wrapper is paid first, so overhead comes straight out of the budget for real content.


In [ ]:
LIMIT = 512
body_tokens = measurements.loc["body_only", "tokens"]

headroom = pd.DataFrame([{
    "format": name,
    "wrapper tokens": measurements.loc[name, "tokens"] - body_tokens,
    "room for content": LIMIT - (measurements.loc[name, "tokens"] - body_tokens),
} for name in FORMATS]).set_index("format")
print(headroom)

# What happens to a longer article, the kind a real knowledge base is full of?
long_article = dict(KNOWLEDGE_BASE[0])
long_article["body"] = " ".join((KNOWLEDGE_BASE[0]["body"] + " ").split() * 6)[:2400]

print()
for name in ["json_indented", "plain_text"]:
    n = count_tokens(FORMATS[name](long_article))
    print(f"long article as {name:<14} {n:>4} tokens ->",
          "TRUNCATED, the end of the body never reaches the model" if n > LIMIT else "fits")

### What it costs at scale

Now lets measure what it costs


In [ ]:
EMBEDDING_RATE = 0.10     # $ per 1M input tokens, replace with your provider's rate
ANSWER_MODEL_RATE = 3.00  # $ per 1M input tokens for the model that writes the answer
CHUNKS = 100_000
QUERIES_PER_MONTH = 100_000
CHUNKS_PER_ANSWER = 5

costs = []
for name in FORMATS:
    tokens = measurements.loc[name, "tokens"]
    index_once = tokens * CHUNKS / 1e6 * EMBEDDING_RATE
    prompts_monthly = tokens * CHUNKS_PER_ANSWER * QUERIES_PER_MONTH / 1e6 * ANSWER_MODEL_RATE
    costs.append({
        "format": name,
        "index once": f"${index_once:,.0f}",
        "prompts / month": f"${prompts_monthly:,.0f}",
        "year one": f"${index_once + prompts_monthly * 12:,.0f}",
    })

pd.DataFrame(costs).set_index("format")

There are two main things to notice from the table.

**Indexing is relatively cheap.** Creating embeddings for 100,000 chunks only costs a few dollars. 

**The bigger cost comes from the prompts.** Every chunk you retrieve has to be added to the prompt that the model uses to generate an answer. For example, if you retrieve five chunks for each question and have 100,000 questions per month, you're adding that extra data to the prompt millions of times. Unlike indexing, which is mostly a one-time cost, this happens over and over for as long as the system is being used.



## What we have established so far

A short recap before you start writing code, since the exercises build on these results.

The same article came to roughly 250 tokens as indented JSON and about 90 as plain text. Most of
that difference is punctuation, field names, and bookkeeping values that a user would never type
into a search box.

Those field names are also identical in every record. `doc_id`, `schema_version` and `access_level`
appear in every chunk, so they give the retrieval system nothing it can use to tell one chunk apart
from another.

Encoders cap their input at a fixed number of tokens, and the wrapper is counted first. A long
article wrapped in JSON can therefore have its ending dropped before the vector is calculated, and
the pipeline reports nothing when that happens.

The cost that accumulates is not the indexing. Embedding a corpus of this size once comes to a few
dollars. The retrieved chunks are then pasted into a prompt every time someone asks a question,
which is where the wrapper gets paid for over and over.

Together these point to a fairly simple rule: embed the fields that carry meaning, and store the
rest alongside the vector.

One thing is still untested. We have shown that plain text is cheaper, but not that it retrieves as
well, and a cheaper chunk that is harder to find would not be an improvement. That is what the
exercises cover. You will write your own formatter, split a record into text and metadata, and then
measure whether the shorter version still returns the right article.


# Exercises

Three exercises. The first two ask you to write the code, and the third asks you to check whether
your optimization actually held up


### Exercise 1 - Write your own formatter

Write a function that turns one record into the string you would embed.

**Requirements:**

1. Keep information a user might say out loud. Someone may search for "reset my password *in
   Cloud Pak for Data*", so the title, product, and version earn their place.
2. Drop bookkeeping nobody queries in natural language: `doc_id`, `author_email`, `schema_version`,
   `source_url`, `access_level`. These are not lost — the next exercise moves them to metadata.
3. No JSON syntax. No braces, no quoted keys.
4. Average under 120 tokens.

Write your version below, then run the checker cell.


In [ ]:
def my_format(rec):
    """Turn one record into the text you want embedded. Return a string."""

    # TODO: return the title, a line of context, and the body, with no JSON syntax.
    #       See the four requirements above this cell.

    return json.dumps(rec)  # DELETE THIS LINE once you have written your version


# When you have edited this cell, run the checker cell below.

In [ ]:
# Checker - run this cell as it is, no changes needed
def check_exercise_1(fn):
    outputs = [fn(rec) for rec in KNOWLEDGE_BASE]
    average = np.mean([count_tokens(o) for o in outputs])
    checks = [
        ("returns a string", all(isinstance(o, str) for o in outputs)),
        ("no JSON syntax", not any('{"' in o or '": ' in o for o in outputs)),
        ("keeps the title", all(r["title"] in o for r, o in zip(KNOWLEDGE_BASE, outputs))),
        ("keeps the body", all(r["body"][:40] in o for r, o in zip(KNOWLEDGE_BASE, outputs))),
        ("keeps the product name", all(r["product"] in o for r, o in zip(KNOWLEDGE_BASE, outputs))),
        ("drops the schema version", not any("1.4.0" in o for o in outputs)),
        ("drops author emails", not any("@example.com" in o for o in outputs)),
        (f"averages under 120 tokens (yours: {average:.0f})", average < 120),
    ]
    for label, passed in checks:
        print(("  PASS  " if passed else "  FAIL  ") + label)
    if all(passed for _, passed in checks):
        print(f"\nAll checks passed. Your format averages {average:.0f} tokens, "
              f"{(1 - average / baseline) * 100:.0f}% below the JSON baseline.")
    else:
        print("\nNot there yet. Read the failing lines above and try again.")

check_exercise_1(my_format)

<details>
    <summary>Click here for Solution</summary>

```python
def my_format(rec):
    header = rec["title"]
    context = f"{rec['product']} {rec['version']} - {rec['category']}"
    return f"{header}\n{context}\n\n{rec['body']}"
```

</details>

There is no single right answer here. A bare `f"{rec['title']}. {rec['body']}"` passes too, and so
does a version that folds the tags in as a comma-separated list. What matters is that the meaning
survives and the syntax does not.


### Exercise 2 - Split the record

Dropping fields from the embedded text does not mean deleting them. Complete `split_record` so that
it returns the two halves: the text you send to the encoder, and the fields you store beside the
vector for exact filtering.


In [ ]:
def split_record(rec):
    """Return {'text': <string to embed>, 'metadata': <dict to store and filter on>}."""
    # TODO: use my_format for the text, and put the non-semantic fields in metadata
    return {"text": "", "metadata": {}}

In [ ]:
# Checker - run this cell as it is, no changes needed
def check_exercise_2(fn):
    result = fn(KNOWLEDGE_BASE[2])
    filterable = ["doc_id", "last_updated", "access_level"]
    checks = [
        ("has 'text' and 'metadata' keys", set(result) == {"text", "metadata"}),
        ("text is not empty", bool(result.get("text"))),
        ("text is not JSON", not result.get("text", "").lstrip().startswith("{")),
        ("metadata is a dictionary", isinstance(result.get("metadata"), dict)),
        ("metadata carries the filterable fields",
         all(k in result.get("metadata", {}) for k in filterable)),
        ("body is not duplicated into metadata", "body" not in result.get("metadata", {})),
    ]
    for label, passed in checks:
        print(("  PASS  " if passed else "  FAIL  ") + label)
    if all(passed for _, passed in checks):
        print("\nGood. Nothing was lost.")

check_exercise_2(split_record)

<details>
    <summary>Click here for Solution</summary>

```python
def split_record(rec):
    for_filtering = ("doc_id", "author", "author_email", "last_updated", "source_url",
                     "access_level", "schema_version", "tags",
                     "product", "version", "category")
    return {
        "text": my_format(rec),
        "metadata": {key: rec[key] for key in for_filtering},
    }
```

</details>

Notice that `product`, `version`, and `category` appear in *both* halves in this solution. A field belongs in the text when users describe it, and in a column when users constrain by it. These fields do both.


### Exercise 3 - Prove it does not hurt retrieval

Shorter chunks are only an improvement if they still return the right article, so this exercise
measures that rather than assuming it.

The measurement works because `QUESTIONS` pairs each of the fourteen questions with the one article
that should answer it. For a given format we embed all fourteen articles, run each question against
them, and record the position of the correct article in the ranked results. Doing that for every
format tells us whether the token savings came at a cost.

Three numbers summarise those positions.

`recall@1` is the share of questions where the correct article came first. A score of 0.71 means 10
of the 14 questions were answered correctly at the top of the list. It is the strictest of the
three, and the one that matters if your interface shows a single result.

`recall@3` is the share where the correct article appeared somewhere in the top three. A score of
0.86 means 12 of 14. For most retrieval-augmented systems this is the more realistic target, since
several chunks are passed to the language model and it works out which one to use.

`MRR` stands for mean reciprocal rank, the average of 1 divided by the position of the correct
article. First place scores 1.0, second 0.5, third 0.33. It picks up changes the recall columns
miss, such as a format that moves the right article from first place to second, which leaves
recall@3 unchanged. It is most useful as a tie-breaker when two formats score the same on recall.

Keep the sample size in mind when you read your results. With fourteen questions each one is worth
about 7 points of recall, so 0.71 against 0.70 is rounding rather than a difference, while 0.71
against 0.57 is two questions.


#### What you already have

You do not need to calculate any of those scores yourself. The `evaluate_retrieval` helper defined
in the Setup section does the embedding, the ranking and the arithmetic, and returns all three in a
dictionary:

    evaluate_retrieval(texts, doc_ids, questions, k=3)

It fits a fresh embedding model on each call, so every format is scored on its own terms and none of
them can borrow information from another.

Four things are already in memory from earlier in the lab:

 - `rendered` - a dictionary mapping each format name to its list of fourteen rendered texts
 - `KNOWLEDGE_BASE` - the articles, in the same order
 - `QUESTIONS` - the fourteen questions and their correct article
 - `my_format` - your own formatter from Exercise 1

Run the cell below to see what one call returns before you write anything.


In [ ]:
doc_ids = [rec["doc_id"] for rec in KNOWLEDGE_BASE]

# One format, scored on its own, to show the shape of the result.
evaluate_retrieval(rendered["plain_text"], doc_ids, QUESTIONS)

#### Your task

Call `evaluate_retrieval` once for each of the four formats in `rendered`, and once more for your own
`my_format`, then collect the results into a single DataFrame with the format name as the index.

Rounding to three decimal places makes the table easier to read.


In [ ]:
scores = []

# TODO 1: loop over rendered.items() and call evaluate_retrieval on each format's texts.
#         Add the format name to the returned dictionary before appending it to scores.
# TODO 2: render the KNOWLEDGE_BASE with your own my_format and score that too,
#         under the name "my_format".
# TODO 3: build a DataFrame from scores, set "format" as the index, and round to 3 places.

quality = None  # replace this
quality

<details>
    <summary>Click here for Solution</summary>

```python
scores = []
for name, texts in rendered.items():
    scores.append({"format": name, **evaluate_retrieval(texts, doc_ids, QUESTIONS)})

mine = [my_format(rec) for rec in KNOWLEDGE_BASE]
scores.append({"format": "my_format", **evaluate_retrieval(mine, doc_ids, QUESTIONS)})

quality = pd.DataFrame(scores).set_index("format").round(3)
quality
```

</details>


### Notice the table output from excersise three

`plain_text` matches both JSON formats while using roughly a third of the tokens. 

Now look at `body_only`. Stripping *everything* — including the title and the product name —
measurably hurts retrieval. That is the guardrail on this lesson: **the goal is removing syntax, not
removing context.** The person asking the question will name the product even when the paragraph does
not, so a chunk still needs enough of a header to be findable on its own.


### When JSON in the embedded text is the right answer

This lab argues for a default, not a law. Keep the structure inside the embedded text when:

 - **The structure is the content.** Code, configuration files, API schemas, and log lines are cases
   where users genuinely search for `"retry_policy"` or a bracket pattern. Strip the syntax and lose accuracy.
 - **The answer must quote fields exactly.** Labelled fields reduce the chance a model swaps two
   values when it writes the answer.
 - **Records are tiny and uniform.** For three short fields, one `field: value` line each is readable
   and costs a few tokens rather than a hundred and eighty.
 - **Something downstream parses the payload.** Store the JSON in metadata and return it with the
   result — just do not embed it.

That last case dissolves the apparent trade-off completely. The embedded text and the stored payload
are different fields. Almost every time someone embeds JSON, they are treating them as one.


### Congratulations! You have completed the lab

You measured the cost of a habit most pipelines never question, rewrote the text that reaches the
encoder, and checked with a retrieval evaluation that the cheaper version is just as good.


## Authors


Sadiq Azmi [Linkedin](https://www.linkedin.com/in/sadiq-azmi/)


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2026-09-17|1.0|Author Name|Create lab: plain text vs JSON in vector storage|


Copyright © 2026 IBM Corporation. All rights reserved.
